In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

fg = pd.read_csv('fear_greed_index.csv')
hd = pd.read_csv('historical_data.csv')

# Document datasets
print(fg.shape)
print(hd.shape)

print(fg.isnull().sum())
print(hd.isnull().sum())

print(fg.duplicated().sum())
print(hd.duplicated().sum())

# Convert timestamps
fg['date'] = pd.to_datetime(fg['date'])
hd['date'] = pd.to_datetime(hd['Timestamp IST'], format='%d-%m-%Y %H:%M').dt.date
hd['date'] = pd.to_datetime(hd['date'])

# merge
merged = hd.merge(fg[['date', 'value', 'classification']], on='date', how='left')

# Key metrics

# daily PnL per account
daily_pnl = hd.groupby(['date', 'Account'])['Closed PnL'].sum().reset_index()

# win rate per account
hd['is_win'] = hd['Closed PnL'] > 0
win_rate = hd.groupby('Account')['is_win'].mean() * 100

# average trade size per account
avg_trade_size = hd.groupby('Account')['Size USD'].mean()

# leverage distribution (size relative to account average)
hd['leverage'] = hd['Size USD'] / hd.groupby('Account')['Size USD'].transform('mean')

# number of trades per day
trades_per_day = hd.groupby('date')['Closed PnL'].count().reset_index()
trades_per_day.columns = ['date', 'num_trades']

# long/short ratio per day
hd['is_long']  = hd['Direction'].str.contains('Long', na=False)
hd['is_short'] = hd['Direction'].str.contains('Short', na=False)
ls_ratio = hd.groupby('date').apply(lambda x: x['is_long'].sum() / max(x['is_short'].sum(), 1)).reset_index()
ls_ratio.columns = ['date', 'long_short_ratio']

#results
print(daily_pnl.head())
print(win_rate.head())
print(avg_trade_size.head())
print(hd['leverage'].describe())
print(trades_per_day.head())
print(ls_ratio.head())